In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib

# 1. Load and Clean Data
adasyn_df = pd.read_csv("adasyn_balanced_1.5M.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour']

# Convert NaN to 0 in targets
adasyn_df[targets] = adasyn_df[targets].fillna(0)

# 2. Feature Engineering (same as before)
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour']/24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour']/24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour'])

# 3. Prepare Data
X = adasyn_df[features]
y = adasyn_df[targets]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Create MLP Pipeline
mlp_pipeline = Pipeline([
    ('features', FunctionTransformer(create_features)),
    ('scaler', StandardScaler()),
    ('clf', MultiOutputClassifier(
        MLPClassifier(
            hidden_layer_sizes=(100, 50),
            activation='relu',
            solver='adam',
            early_stopping=True,
            random_state=42,
            verbose=False
        )
    ))
])

# 5. Hyperparameter Tuning (reduced grid for efficiency)
param_grid = {
    'clf__estimator__hidden_layer_sizes': [(50,), (100,50)],
    'clf__estimator__alpha': [0.0001, 0.001],
    'clf__estimator__learning_rate_init': [0.001, 0.01]
}

# 6. Train with subsample (MLPs are computationally intensive)
tune_sample = min(50000, len(X_train))  # Smaller sample for tuning
X_tune = X_train.sample(tune_sample, random_state=42)
y_tune = y_train.loc[X_tune.index]

grid_search = GridSearchCV(
    mlp_pipeline,
    param_grid,
    cv=3,  # Fewer folds for speed
    scoring='f1_weighted',  # Default to weighted F1 for each output
    verbose=2,
    n_jobs=1
)

print("Training MLP...")
grid_search.fit(X_tune, y_tune)

# 7. Final Training
best_mlp = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

# Train on full data (optional - can be very slow)
# best_mlp.fit(X_train, y_train)

# 8. Evaluation
y_pred = best_mlp.predict(X_test)
print("\nTest Performance:")
for i, target in enumerate(targets):
    print(f"\n{target}:")
    print(classification_report(y_test.iloc[:,i], y_pred[:,i]))

# 9. Save Model
joblib.dump(best_mlp, 'mlp_oversampled_tuned.pkl')

Training MLP...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END clf__estimator__alpha=0.0001, clf__estimator__hidden_layer_sizes=(50,), clf__estimator__learning_rate_init=0.001; total time=  33.3s
[CV] END clf__estimator__alpha=0.0001, clf__estimator__hidden_layer_sizes=(50,), clf__estimator__learning_rate_init=0.001; total time=  32.9s
[CV] END clf__estimator__alpha=0.0001, clf__estimator__hidden_layer_sizes=(50,), clf__estimator__learning_rate_init=0.001; total time=  26.8s
[CV] END clf__estimator__alpha=0.0001, clf__estimator__hidden_layer_sizes=(50,), clf__estimator__learning_rate_init=0.01; total time=  42.9s
[CV] END clf__estimator__alpha=0.0001, clf__estimator__hidden_layer_sizes=(50,), clf__estimator__learning_rate_init=0.01; total time=  34.4s
[CV] END clf__estimator__alpha=0.0001, clf__estimator__hidden_layer_sizes=(50,), clf__estimator__learning_rate_init=0.01; total time=  37.8s
[CV] END clf__estimator__alpha=0.0001, clf__estimator__hidden_layer_sizes=(

['mlp_oversampled_tuned.pkl']

In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, precision_recall_curve, make_scorer, recall_score
import joblib

# 1. Load ADASYN-oversampled data
df = pd.read_csv("adasyn_balanced_1.5M.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour']
#filling nans with 0
df[targets] = df[targets].fillna(0)
# 2. Enhanced Feature Engineering
def create_enhanced_features(X):
    X = X.copy()
    # Cyclical time features
    X['hour_sin'] = np.sin(2 * np.pi * X['hour']/24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour']/24)
    
    # Behavioral ratios
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    
    # New interaction features
    X['activity_eat_ratio'] = X['ACTIVITY_LEVEL'] / (X['EAT'] + 1e-6)
    X['rest_alley_ratio'] = X['REST'] / (X['IN_ALLEYS'] + 1e-6)
    
    return X.drop(columns=['hour'])

# 3. Prepare data
X = df[features]
y = df[targets]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Custom scoring for CV (focus on disease recall)
def disease_recall_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    recalls = []
    for i in range(y.shape[1]):
        recalls.append(recall_score(y.iloc[:,i], y_pred[:,i], pos_label=1, zero_division=0))
    return np.mean(recalls)

scorer = make_scorer(disease_recall_scorer, greater_is_better=True)

# 5. Build pipeline
mlp = MLPClassifier(early_stopping=True, random_state=42)
pipeline = Pipeline([
    ('features', FunctionTransformer(create_enhanced_features)),
    ('scaler', StandardScaler()),
    ('clf', MultiOutputClassifier(mlp))
])

# 6. Hyperparameter grid
param_grid = {
    'clf__estimator__hidden_layer_sizes': [(128,), (128, 64)],
    'clf__estimator__alpha': [0.0001, 0.001],
    'clf__estimator__learning_rate_init': [0.001, 0.01],
    'clf__estimator__batch_size': [256, 512]
}

# 7. Create smaller tuning sample (for faster CV)
tune_sample = min(100000, len(X_train))
X_tune = X_train.sample(tune_sample, random_state=42)
y_tune = y_train.loc[X_tune.index]

# 8. Configure CV and GridSearch
cv = KFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring=scorer,
    verbose=3,
    n_jobs=1  # MLP doesn't parallelize well
)

# 9. Run hyperparameter tuning
print("Starting GridSearchCV...")
grid_search.fit(X_tune, y_tune)

# 10. Train final model with best params
best_model = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

# Fit on full data
best_model.fit(X_train, y_train)

# 11. Threshold optimization
optimal_thresholds = {}
for i, target in enumerate(targets):
    y_probs = best_model.predict_proba(X_test)[i][:,1]
    precision, recall, thresholds = precision_recall_curve(y_test[target], y_probs)
    
    # Find threshold that gives at least 70% recall
    target_recall = 0.7
    viable_thresholds = thresholds[recall[:-1] >= target_recall]
    optimal_thresholds[target] = viable_thresholds[0] if len(viable_thresholds) > 0 else 0.5

# 12. Evaluate
y_pred_adjusted = np.zeros_like(y_test)
for i, target in enumerate(targets):
    y_probs = best_model.predict_proba(X_test)[i][:,1]
    y_pred_adjusted[:,i] = (y_probs > optimal_thresholds[target]).astype(int)

print("\nOptimized Performance:")
for i, target in enumerate(targets):
    print(f"\n{target}:")
    print(classification_report(y_test[target], y_pred_adjusted[:,i]))

# 13. Save everything
joblib.dump({
    'model': best_model,
    'thresholds': optimal_thresholds,
    'cv_results': grid_search.cv_results_
}, 'tuned_mlp_model.pkl')

Starting GridSearchCV...
Fitting 3 folds for each of 16 candidates, totalling 48 fits


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:982: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 971, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 279, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 376, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[CV 1/3] END clf__estimator__alpha=0.0001, clf__estimator__batch_size=256, clf__estimator__hidden_layer_sizes=(128,), clf__estimator__learning_rate_init=0.001;, score=nan total time= 1.9min


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:982: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 971, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 279, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 376, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[CV 2/3] END clf__estimator__alpha=0.0001, clf__estimator__batch_size=256, clf__estimator__hidden_layer_sizes=(128,), clf__estimator__learning_rate_init=0.001;, score=nan total time= 1.3min


...

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:982: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 971, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 279, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 376, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[CV 2/3] END clf__estimator__alpha=0.001, clf__estimator__batch_size=512, clf__estimator__hidden_layer_sizes=(128, 64), clf__estimator__learning_rate_init=0.01;, score=nan total time=  23.7s


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:982: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 971, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 279, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 376, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[CV 3/3] END clf__estimator__alpha=0.001, clf__estimator__batch_size=512, clf__estimator__hidden_layer_sizes=(128, 64), clf__estimator__learning_rate_init=0.01;, score=nan total time=  26.2s

Best Parameters: {'clf__estimator__alpha': 0.0001, 'clf__estimator__batch_size': 256, 'clf__estimator__hidden_layer_sizes': (128,), 'clf__estimator__learning_rate_init': 0.001}

Optimized Performance:

oestrus:
              precision    recall  f1-score   support

         0.0       1.00      0.00      0.00    225571
         1.0       0.25      1.00      0.40     74429

    accuracy                           0.25    300000
   macro avg       0.62      0.50      0.20    300000
weighted avg       0.81      0.25      0.10    300000


calving:
              precision    recall  f1-score   support

         0.0       1.00      0.00      0.00    224822
         1.0       0.25      1.00      0.40     75178

    accuracy                           0.25    300000
   macro avg       0.63      0.50      0.2

['tuned_mlp_model.pkl']